In [4]:
import sxtwl
import argparse
import collections
import pprint
import datetime

from lunar_python import Lunar
from colorama import init

from datas import *
from sizi import summarys
from common import *
from yue import months

In [19]:

year = 1996
month = 9
day = 17
time = 3
gender = "F"
day = sxtwl.fromSolar(year, month, day)

gz = day.getHourGZ(time)
yTG = day.getYearGZ()
mTG = day.getMonthGZ()
dTG  = day.getDayGZ()

In [25]:
Gans = collections.namedtuple("Gans", "year month day time")
Zhis = collections.namedtuple("Zhis", "year month day time")
gans = Gans(year=Gan[yTG.tg], month=Gan[mTG.tg], 
            day=Gan[dTG.tg], time=Gan[gz.tg])
zhis = Zhis(year=Zhi[yTG.dz], month=Zhi[mTG.dz], 
            day=Zhi[dTG.dz], time=Zhi[gz.dz])


me = gans.day
month = zhis.month
alls = list(gans) + list(zhis)
zhus = [item for item in zip(gans, zhis)]
print(alls)
print(zhus)

gan_shens = []
for seq, item in enumerate(gans):    
    if seq == 2:
        gan_shens.append('--')
    else:
        gan_shens.append(ten_deities[me][item])
print(gan_shens)

['丙', '丁', '丁', '壬', '子', '酉', '巳', '寅']
[('丙', '子'), ('丁', '酉'), ('丁', '巳'), ('壬', '寅')]
['劫', '比', '--', '官']


In [15]:
zhi_shens = [] # 地支的主气神
for item in zhis:
    d = zhi5[item]
    zhi_shens.append(ten_deities[me][max(d, key=d.get)])
print(zhi_shens)
shens = gan_shens + zhi_shens

['杀', '才', '劫', '印']


In [23]:
scores = {"金":0, "木":0, "水":0, "火":0, "土":0}
gan_scores = {"甲":0, "乙":0, "丙":0, "丁":0, "戊":0, "己":0, "庚":0, "辛":0,
              "壬":0, "癸":0}   

for item in gans:  
    scores[gan5[item]] += 5
    gan_scores[item] += 5


for item in list(zhis) + [zhis.month]:  
    for gan in zhi5[item]:
        scores[gan5[gan]] += zhi5[item][gan]
        gan_scores[gan] += zhi5[item][gan]


# 计算八字强弱
# 子平真诠的计算
weak = True
me_status = []
for item in zhis:
    me_status.append(ten_deities[me][item])
    if ten_deities[me][item] in ('长', '帝', '建'):
        weak = False
        

if weak:
    if shens.count('比') + me_status.count('库') >2:
        weak = False

gan_scores


# 网上的计算
me_attrs_ = ten_deities[me].inverse
strong = gan_scores[me_attrs_['比']] + gan_scores[me_attrs_['劫']] \
    + gan_scores[me_attrs_['枭']] + gan_scores[me_attrs_['印']]


temps_scores = temps[gans.year] + temps[gans.month] + temps[me] + temps[gans.time] + temps[zhis.year] + temps[zhis.month]*2 + temps[zhis.day] + temps[zhis.time]
print("\033[1;36;40m五行分数", scores, '  八字强弱：', strong, "通常>29为强，需要参考月份、坐支等", "weak:", weak)



五行分数 {'金': 17, '木': 5, '水': 13, '火': 22, '土': 3}   八字强弱： 27 通常>29为强，需要参考月份、坐支等 weak: False


In [20]:
# 计算大运
seq = Gan.index(gans.year)
if gender == "F":
    if seq % 2 == 0:
        direction = -1
    else:
        direction = 1
else:
    if seq % 2 == 0:
        direction = 1
    else:
        direction = -1

dayuns = []
gan_seq = Gan.index(gans.month)
zhi_seq = Zhi.index(zhis.month)
for i in range(12):
    gan_seq += direction
    zhi_seq += direction
    dayuns.append(Gan[gan_seq%10] + Zhi[zhi_seq%12])

print("--------------------")
print("大运:", dayuns)
print("--------------------")

--------------------
大运: ['丙申', '乙未', '甲午', '癸巳', '壬辰', '辛卯', '庚寅', '己丑', '戊子', '丁亥', '丙戌', '乙酉']
--------------------


In [24]:
birthday = datetime.date(day.getSolarYear(), day.getSolarMonth(), day.getSolarDay()) 
count = 0
for i in range(30):    
    #print(birthday)
    day_ = sxtwl.fromSolar(birthday.year, birthday.month, birthday.day)
    #if day_.hasJieQi() and day_.getJieQiJD() % 2 == 1
    if day_.hasJieQi() and day_.getJieQi() % 2 == 1:
            break
        #break        
    birthday += datetime.timedelta(days=direction)
    count += 1

ages = [(round(count/3 + 10*i), round(year + 10*i + count//3)) for i in range(12)]
print(list(zip(ages, dayuns)))

[((3, 1999), '丙申'), ((13, 2009), '乙未'), ((23, 2019), '甲午'), ((33, 2029), '癸巳'), ((43, 2039), '壬辰'), ((53, 2049), '辛卯'), ((63, 2059), '庚寅'), ((73, 2069), '己丑'), ((83, 2079), '戊子'), ((93, 2089), '丁亥'), ((103, 2099), '丙戌'), ((113, 2109), '乙酉')]
